# Week 7 — Observability, privacy, reliability, cost, and GenAIOps

Trace model and tool behavior with OpenTelemetry/Application Insights while minimizing customer data. A release includes its model, prompt, tools, knowledge assets, evaluation evidence, operational limits, and rollback target.

In [ ]:
import importlib.util
import sys
from pathlib import Path

curriculum_root = next(
    candidate
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base, base / "examples" / "foundry-curriculum")
    if (candidate / "notebook_setup.py").is_file()
)
spec = importlib.util.spec_from_file_location(
    "foundry_curriculum_setup", curriculum_root / "notebook_setup.py"
)
helpers = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helpers
spec.loader.exec_module(helpers)
session = helpers.load_session(curriculum_root)
session.safe_summary()

## An operations contract is a set of promises with owners

The next cell is not documentation. Each field is a promise someone can be
paged about, and the four groups answer four different questions:

- **`slos`** — what "working" means numerically, including cost. An agent that
  meets its latency SLO at four times the budgeted cost per success is failing.
- **`trace_policy`** — what you are allowed to keep. Note `prompt_capture` is
  off by default: the most useful debugging field is also the one most likely
  to contain customer data.
- **`failure_drills`** — the dependency failures you have *rehearsed*, not the
  ones you can imagine.
- **`degraded_mode`** — what the system does instead of failing open.

The last one is the field teams skip. Without it, "the model is throttled"
resolves to whatever the code happens to do.

In [ ]:
operations_contract = {
    "slos": {
        "task_success_minimum": 0.85,
        "availability_minimum": 0.995,
        "p95_latency_seconds_maximum": 8.0,
        "cost_per_success_budget": "set-from-approved-pricing-source",
    },
    "trace_policy": {
        "prompt_capture": "off-by-default",
        "tool_argument_capture": "allow-listed-fields-only",
        "retention_days": "set-by-data-policy",
        "access_group": "group:ai-platform-operators",
    },
    "failure_drills": [
        "model throttling",
        "search unavailable",
        "tool timeout",
        "model version retirement",
    ],
    "degraded_mode": "answer only from approved static guidance",
}
operations_contract

### What you just saw

A dict with no runtime behind it — deliberately, because every field here has
to be agreed before it can be instrumented. Two of them are directly checkable
once telemetry exists.

`p95_latency_seconds_maximum` and `availability_minimum` both come from spans
already flowing to Application Insights. The AgentOps Accelerator reads that
same connection to check whether the signals a gate depends on are actually
arriving:

```bash
agentops telemetry validate   # is APPLICATIONINSIGHTS_CONNECTION_STRING live?
agentops telemetry preview    # what traces and eval events are landing?
```

`telemetry validate` answers a question that is easy to assume: *is the
observability the release gate depends on connected at all?* A gate reading
an empty telemetry source reports health, not silence.

### Change this and re-run

Set `prompt_capture` to `"on"` and read the `trace_policy` block again as if
you were the data-protection reviewer. `retention_days` and `access_group`
stop being administrative fields and become the entire control. That is the
trade: the debugging signal you want most is the one that turns a trace store
into a customer-data store.

## The loop that keeps the evaluation set honest

Notebook 05's 20 cases are static. Production is not. Every week the gate runs
against a set that reflects what you imagined at the start, while real traffic
drifts somewhere else.

The AgentOps Accelerator closes that gap with `agentops eval promote-traces`,
which turns production traces into candidate dataset rows:

```text
monitor production  ->  promote reviewed traces  ->  re-evaluate  ->  fresh evidence
```

The accelerator's own framing is the important part: **promotion is
review-first.** Auto-promoting traces into a gate lets production behaviour
redefine the standard the gate enforces — including the behaviour you would
have wanted to catch. A trace is a candidate case, and a human decides whether
it becomes one, and what the expected answer is.

The next cell models that decision so the rule is executable rather than
aspirational.

In [ ]:
def promote(trace, *, reviewer=None, expected=None):
    """Turn one production trace into a dataset row, or refuse to."""

    if trace.get("contains_customer_data"):
        # Promotion copies the trace into a file that lives in Git for years.
        return None, "redact before promoting: trace carries customer data"
    if reviewer is None:
        return None, "a reviewer must accept the case"
    if not expected:
        # The trace records what the agent *did*. Promoting it unchanged makes
        # current behaviour the standard, so the case can never fail.
        return None, "a reviewer must supply the expected answer"
    return (
        {"input": trace["input"], "expected": expected, "source_trace": trace["id"]},
        f"promoted by {reviewer}",
    )


REVIEWER = "group:ai-platform-operators"

traces = [
    {
        "id": "t-1001",
        "input": "How do I rotate the key?",
        "contains_customer_data": False,
    },
    {
        "id": "t-1002",
        "input": "Summarise account 55-2210",
        "contains_customer_data": True,
    },
]

unreviewed, why_unreviewed = promote(traces[0])
observed_only, why_observed = promote(traces[0], reviewer=REVIEWER)
accepted, why_accepted = promote(
    traces[0],
    reviewer=REVIEWER,
    expected="Rotate through the approved platform process; never store the key.",
)
redacted, why_redacted = promote(traces[1], reviewer=REVIEWER)

assert unreviewed is None
assert observed_only is None  # a reviewer alone is not enough
assert redacted is None  # customer data blocks promotion outright
assert accepted is not None and accepted["source_trace"] == "t-1001"

{
    "unreviewed": why_unreviewed,
    "reviewed_without_expected": why_observed,
    "customer_data": why_redacted,
    "accepted": why_accepted,
}

### What you just saw

Four attempts, one promotion. The rejection that matters most is the second
one: a named reviewer accepted the trace and it still did not become a case,
because no expected answer was supplied.

That is the difference between a regression case and a recording. A row whose
`expected` is copied from what the agent produced can never fail — it locks in
today's behaviour as tomorrow's standard, and it does so invisibly, because
the gate stays green while the set stops testing anything.

The `contains_customer_data` rejection is the other half. A promoted trace is
copied into a file that lives in Git for years, which is a different retention
decision from the one `trace_policy` made about the trace store above.

### Change this and re-run

Have `promote` default `expected` to `trace.get("output")`. Every rejection
except the customer-data one disappears and the loop becomes fully automatic —
which is exactly the failure mode. Convenience here costs you the ability to
detect regression at all.

## A release is everything that can change the answer

The next cell lists what has to be pinned for a result to be reproducible.
The unfamiliar entries are the informative ones: `embedding_and_chunking_versions`
and `knowledge_index_version` change retrieval behaviour without a single line
of application code changing, and `prompt_digest` exists because a prompt
edited in a portal leaves no commit behind.

`rollback_target` is last and is the one that makes the rest actionable. A
manifest you cannot roll back to is a description of an incident, not a
control.

In [ ]:
release_manifest = {
    "foundry_project": session.project_endpoint,
    "logical_model": session.logical_model,
    "model_deployment": session.deployment,
    "model_version": "record-before-release",
    "prompt_digest": "record-before-release",
    "tool_schema_versions": [],
    "knowledge_index_version": "record-before-release",
    "embedding_and_chunking_versions": "record-before-release",
    "evaluation_run_id": None,
    "red_team_result_id": None,
    "rollback_target": None,
}
release_manifest

### What you just saw

A manifest mostly full of `"record-before-release"` — honest placeholders for
values only a real release can supply.

The AgentOps Accelerator produces a machine-readable version of this same idea
rather than asking you to hand-maintain it:

```bash
agentops doctor --evidence-pack --severity-fail critical
```

which writes:

| path | what it holds |
|---|---|
| `.agentops/results/latest/results.json` | scores, machine-readable |
| `.agentops/results/latest/report.md` | the same run, for a human |
| `.agentops/agent/report.md` | Doctor findings across quality, reliability, security |
| `.agentops/release/latest/evidence.{json,md}` | the readiness summary |

Compare the two side by side. `release_manifest` records *what was released*;
the evidence pack records *why it was allowed to be*. A release needs both,
and the pair is what a reviewer six months from now actually reads.

Doctor's severity gate has three settings — `critical` (default), `warning`,
`none` — and the choice is a policy decision, not a tuning knob. `none` makes
Doctor advisory, which is the right starting point for an existing system and
the wrong permanent state.

### Change this and re-run

Set `rollback_target` to a model *alias* rather than an immutable version.
Nothing in the notebook complains. Now ask what the alias pointed at during
the incident you are rolling back from — aliases move, and the manifest has
quietly stopped identifying anything.

## Exit criteria

Demonstrate a throttled or failed dependency, verify degraded behavior, query
latency and errors without exposing prompt data, and test rollback to a
known-good immutable version. Then show one production trace promoted into the
evaluation set with a named reviewer and a reviewer-written expected answer,
and the evidence pack that the resulting run produced.

Do not assume Hosted Agent weighted traffic splitting is available.